# S1.1 · Data, variables and measurement

*Phase S, statistics and probability · Part 1, describing data · lesson 1 of 10 · about 2.5 hours*

Part 1 is about describing data: counting, drawing, summing up. Before any of that, a careful analyst asks four questions of every table. What does one row stand for? What kind of value does each column hold? How were the values measured? What is missing, and why? This lesson builds that habit on the seven tables of *Bibliothecae\.it*, and it ends with a question for your PhD: how could "reuse" be measured?

**Run this lesson:** [in your browser (JupyterLite)](https://sbrzt.github.io/personal-course/lite/lab/index.html?path=lessons/phase-s/s1-1-data-variables-measurement.ipynb) · [in Google Colab](https://colab.research.google.com/github/sbrzt/personal-course/blob/main/lessons/phase-s/s1-1-data-variables-measurement.ipynb) · on your computer (see *How this course works* on the course site)

**By the end you can**

- say what one row of a table stands for, and tell observations from variables;
- name the type of a variable (nominal, ordinal, interval or ratio; discrete or continuous), and say what can be computed with it;
- turn text into numbers, and avoid the traps of numbers stored as text;
- recognise a tidy table, and tidy an untidy column;
- tell a population from a sample;
- explain operationalisation, validity and reliability, and apply them to a measure of your own;
- find missing values, and ask why they are missing.

Exercises, hints, solutions and sliders work as in Part 0. Run the setup cell first.

In [ ]:
# ⚙️ Setup: run this cell first
import sys, pathlib, subprocess

if sys.platform == "emscripten":            # in the browser (JupyterLite)
    import piplite
    await piplite.install(["personal-course"])
elif "google.colab" in sys.modules:         # in Google Colab
    repo = "git+https://github.com/sbrzt/personal-course"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"personal-course @ {repo}"], check=True)
else:                                       # on your computer
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:    # find the course folder
        if (folder / "course" / "__init__.py").exists():
            sys.path.insert(0, str(folder))
            break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course import bibliothecae, data, plots, stats
from course.widgets import explore
from course.checks import s1_1 as check

plots.use_style()
bib = data.bibliothecae()          # the seven tables of the journal's data
print("Ready.")

## 1 · What does one row stand for?

A library keeps a register of its loans:

| reader | book | date out | days kept |
| --- | --- | --- | --- |
| Anna | *Il nome della rosa* | 2 March | 14 |
| Marco | *Le città invisibili* | 2 March | 21 |
| Anna | *Le città invisibili* | 25 March | 7 |

Each row is one loan. Anna appears twice because she borrowed two books, and *Le città invisibili* appears twice because it was borrowed twice. The thing that one row stands for is the table's **unit of observation**, and each row is one **observation**. Each column is a **variable**: a property measured on every observation. Each cell holds one **value**.

The first question to ask of any table is *what does one row stand for?* Get it wrong and every count is wrong. To ask "how many books does a reader borrow?", the unit of analysis is the reader, not the loan, so you must first gather each reader's loans: Anna borrowed 2, Marco 1.

Statistics writes a table of numbers with two subscripts:

$$x_{ij}$$

| Symbol | Read it as | Means |
| --- | --- | --- |
| $i$ | "i" | the number of the row: which observation, from 1 to n |
| $j$ | "j" | the number of the column: which variable, from 1 to p |
| $x_{ij}$ | "x i j" | the value of the j-th variable for the i-th observation |
| $n$, $p$ | "n", "p" | the number of observations (rows) and of variables (columns) |

Read aloud: *"x i j is the value in row i, column j."* A table with n rows and p columns is called "n by p", written $n \times p$. In the loan register, $n = 3$ and $p = 4$, and $x_{34} = 7$: the third loan was kept for 7 days. In pandas, `table.iloc[2, 3]` gives the same cell, because Python counts from 0 (S0.1).

**In the data.** *Bibliothecae\.it* comes as seven tables. Four describe things: articles, their authors, the references in their reference lists, and the authors of those references. Three are *link* tables, whose rows connect one thing to another:

In [ ]:
boxes = {"article_authors": (0, 1), "article_author": (1, 1), "articles": (2, 1),
         "article_reference": (3, 1), "references": (4, 1),
         "reference_author": (4, 0), "reference_authors": (3, 0)}
links = ["article_author", "article_reference", "reference_author"]
edges = [("article_authors", "article_author"), ("article_author", "articles"),
         ("articles", "article_reference"), ("article_reference", "references"),
         ("references", "reference_author"),
         ("reference_author", "reference_authors")]

fig, ax = plt.subplots(figsize=(9, 2.8))
for one, other in edges:
    (x1, y1), (x2, y2) = boxes[one], boxes[other]
    ax.plot([x1, x2], [y1, y2], color=plots.MUTED, zorder=1)
for name, (x, y) in boxes.items():
    fill = "white" if name in links else "#dbe8f8"
    ax.text(x, y, f"{name}\n{len(getattr(bib, name)):,} rows", ha="center",
            va="center", fontsize=9, zorder=2,
            bbox=dict(boxstyle="round,pad=0.5", fc=fill, ec=plots.BLUE))
ax.set(xlim=(-0.6, 4.6), ylim=(-0.5, 1.5))
ax.axis("off")
plt.show()

Why link tables? An article can have several authors, and an author can write several articles. Each pair gets a row of its own in `article_author`, so that no cell ever has to hold a list. Here is what one row stands for in each table:

In [ ]:
one_row = {
    "articles": "an article of the journal",
    "article_authors": "an author of articles, with sex and affiliations",
    "references": "a reference in an article's reference list",
    "reference_authors": "an author of cited works, with a local identifier",
    "article_author": "a link: this article has this author",
    "article_reference": "a link: this article cites this reference",
    "reference_author": "a link: this reference has this author",
}
pd.DataFrame({"rows (n)": [len(getattr(bib, t)) for t in one_row],
              "columns (p)": [getattr(bib, t).shape[1] for t in one_row],
              "one row is": list(one_row.values())}, index=list(one_row))

### ✏️ Exercise 1 · One row, one link?

In `bib.article_reference`, each row should be one link between an article (column `id`) and a reference it cites (column `reference`). Compute `n_links`, the number of rows; `n_distinct`, the number of different rows, counting duplicated rows once; and `n_citing`, the number of different articles that appear in the table.

In [ ]:
n_links = ...
n_distinct = ...
n_citing = ...

check.ex1(n_links, n_distinct, n_citing)

<details>
<summary>💡 Hint</summary>

`len(table)` counts rows, and `table.drop_duplicates()` keeps one copy of each row. `table["id"].nunique()` counts the different values in a column.

</details>

<details>
<summary>✅ Solution</summary>

```python
n_links = len(bib.article_reference)
n_distinct = len(bib.article_reference.drop_duplicates())
n_citing = bib.article_reference["id"].nunique()

check.ex1(n_links, n_distinct, n_citing)
```

</details>

## 2 · Kinds of variables

A library catalogue records, for each item, its format (book, journal, map), its condition (good, fair, poor), the year it was printed and its number of pages. These four variables hold four different kinds of value, and each kind allows different calculations. The psychologist S. S. Stevens named the four kinds, or *scales of measurement*, in 1946 ([Stevens, 1946](https://doi.org/10.1126/science.103.2684.677)):

| Type | Its values are | You can | Examples |
| --- | --- | --- | --- |
| **nominal** | names of categories, in no order | say whether two values are the same, and count them | format, language, section, sex |
| **ordinal** | categories in an order | also say which of two values comes first | condition, a rank, first or second issue |
| **interval** | numbers with equal steps but no true zero | also add and subtract | calendar year, temperature in °C |
| **ratio** | numbers with a true zero | also multiply and divide: "twice as many" | pages, references, citations, age |

Each type allows everything the types above it allow, plus one more operation.

- **Nominal.** Italian and English are different languages, but neither is "more" than the other. Their order in a table is up to you.
- **Ordinal.** "Good" is better than "fair", which is better than "poor", but the steps need not be equal. Ranks give the order, not the distances: in the *Commedia* (S0.3), the word ranked 1st occurs about 1.1 times as often as the word ranked 2nd, while the word ranked 2nd occurs about 1.4 times as often as the word ranked 3rd.
- **Interval.** From 2012 to 2024 is 12 years, the same as from 1500 to 1512: differences make sense. But the year 2000 is not twice the year 1000, because the year 0 is a convention, not "no time".
- **Ratio.** An article of 40 pages is twice as long as one of 20, because 0 pages really means no pages.

A second distinction cuts across the four. **Discrete** variables take separate values, like the counts 0, 1, 2 (references, citations, pages, authors). **Continuous** variables can take any value in a range, like a length of time or a sum of money, even when we round them. Most bibliometric variables are counts: ratio variables that are discrete.

Beware of digits that are only names. A DOI, a Wikidata identifier, a volume number and a postcode are written with digits, but adding them up means nothing. The type of a variable depends on what its values mean, not on how they are written.

Each type has graphs that suit it. Here are one variable of each type from the journal; you will meet each kind of graph properly in S1.2 and S1.3.

In [ ]:
year = bib.articles["date"].str[:4].astype(int)
first_last = bib.articles["pages"].str.split("-", expand=True).astype(int)
page_counts = first_last[1] - first_last[0] + 1

fig, axes = plt.subplots(2, 2, figsize=(9, 5.6))
bib.articles["section"].value_counts().plot.barh(ax=axes[0, 0], color=plots.BLUE)
axes[0, 0].set(title="nominal: section", xlabel="articles")
bib.articles["issue"].value_counts().loc[["1", "2", "1-2"]].plot.bar(
    ax=axes[0, 1], color=plots.BLUE, rot=0)
axes[0, 1].set(title="ordinal: issue", ylabel="articles")
year.value_counts().sort_index().plot.bar(ax=axes[1, 0], color=plots.BLUE)
axes[1, 0].set(title="interval: year", xlabel="year", ylabel="articles")
axes[1, 1].hist(page_counts, bins=np.arange(0, 210, 10), color=plots.BLUE)
axes[1, 1].set(title="ratio: pages", xlabel="pages", ylabel="articles")
fig.tight_layout()
plt.show()

The sections could be listed in any order: here they are sorted by size. The issues have an order, plus a third value, "1-2", for nine articles in double issues: real data rarely fits a type perfectly. The years are evenly spaced, and the page counts start from a true zero.

### ✏️ Exercise 2 · What type is each variable?

For each variable, choose its type: N (nominal), O (ordinal), I (interval) or R (ratio).

- **A** the language of an article
- **B** the year an article was published
- **C** a word's rank in the frequency list of the *Commedia*
- **D** the number of references in an article
- **E** the section of the journal an article appears in
- **F** the age of a cited work, in years, when it was cited
- **G** the number of pages of an article
- **H** whether an article appeared in the first or the second issue of its volume

Set `answer` to eight letters, one per variable in order, for example `"NNNNNNNN"`.

In [ ]:
answer = ...

check.ex2(answer)

<details>
<summary>💡 Hint</summary>

For each variable, ask the questions of the table in order. Do its values have an order? Are the steps between them equal? Does 0 mean "none of it", so that "twice as much" makes sense? An age is a length of time, so an age of 0 means "just published".

</details>

<details>
<summary>✅ Solution</summary>

```python
answer = "NIORNRRO"

check.ex2(answer)
```

A year is interval, because the year 0 is a convention, but an age is ratio, because an age of 0 really means no time has passed: a work cited at 40 years old is twice as old as one cited at 20.

</details>

## 3 · What the computer thinks a value is

The course loads every value of the seven tables as text, a *string*, exactly as it appears in the files, so that nothing is changed behind your back. Text is fine for nominal variables. To compute with numbers or dates, you convert them first. Until you do, text plays tricks:

In [ ]:
print(bib.articles.dtypes.value_counts())     # 'str' or 'object': text, here
volumes = bib.articles["volume"]
print("as text:   ", sorted(volumes.unique()))
print("as numbers:", sorted(volumes.astype(int).unique().tolist()))

Text is sorted character by character, as in a dictionary: "10" comes before "9" because "1" comes before "9". So the largest volume, as text, is "9". Three ways of converting:

- `.astype(int)` turns clean text such as `"14"` into the number 14, and fails on anything else, which is a useful alarm.
- `pd.to_numeric(column, errors="coerce")` turns what it can into numbers and the rest into `NaN`, *not a number*, pandas' mark for a missing number (section 7).
- `pd.to_datetime(column)` turns text such as `"2014-07-01T00:00:00Z"` into dates.

**Two numbers in one cell.** The `pages` column holds the first and last page of each article, as in `"137-184"`. To count the pages, split the text at the hyphen, convert both parts, and subtract:

$$\text{pages} = \text{last} - \text{first} + 1$$

Read aloud: *"the number of pages is the last page minus the first, plus one."* The plus one is easy to forget. **By hand:** pages 137 to 184 are $184 - 137 + 1 = 48$ pages, not 47, because both the first and the last page count: an article on page 3 alone has $3 - 3 + 1 = 1$ page. Counting the posts of a fence, not the gaps between them, is a classic source of "off by one" errors.

**A date that belongs to something else.** The 321 articles have only 26 different dates:

In [ ]:
dates = pd.to_datetime(bib.articles["date"]).dt.date
print(f"{dates.nunique()} different dates for {len(dates)} articles")
print(dates.value_counts().head(4))

The date in the articles table belongs to the issue, not to the article: every article of an issue repeats it. Its unit is not the table's unit, which matters if you ever ask what day of the year articles appear.

### ✏️ Exercise 3 · Text that should be numbers

Compute `max_text`, the largest value of `bib.articles["volume"]` as text, and `max_number`, its largest value as a number. Then compute `n_pages`, the number of pages of every article (a column of 321 numbers), and `total_pages`, the total number of pages the journal has published.

In [ ]:
max_text = ...
max_number = ...
n_pages = ...
total_pages = ...

check.ex3(max_text, max_number, n_pages, total_pages)

<details>
<summary>💡 Hint</summary>

`.max()` works on text and on numbers. For the pages, `bib.articles["pages"].str.split("-", expand=True)` gives a table with the first pages in column 0 and the last pages in column 1; convert it with `.astype(int)`, then subtract, and remember the plus one.

</details>

<details>
<summary>✅ Solution</summary>

```python
max_text = bib.articles["volume"].max()
max_number = bib.articles["volume"].astype(int).max()
first_last = bib.articles["pages"].str.split("-", expand=True).astype(int)
n_pages = first_last[1] - first_last[0] + 1
total_pages = n_pages.sum()

check.ex3(max_text, max_number, n_pages, total_pages)
```

</details>

## 4 · Tidy tables

A table can hold the same information in many shapes, and some shapes are much easier to work with than others. The statistician Hadley Wickham called the easiest shape **tidy**: "each variable is a column, each observation is a row, and each type of observational unit is a table" ([Wickham, 2014](https://doi.org/10.18637/jss.v059.i10)).

The seven tables are mostly tidy: the link tables exist so that each row holds one link. Two columns break the rules, because each of their cells holds several values:

In [ ]:
bib.articles[["id", "keywords"]].head(3)

The `keywords` column holds a list in each cell, separated by `"; "`. To count how often each keyword is used, you need one row per article and keyword. `.str.split("; ")` turns each cell into a list, and `.explode()` gives each item of each list a row of its own:

In [ ]:
keywords = bib.articles.set_index("id")["keywords"].str.split("; ").explode()
keywords.head(8)

Now each row is one pair, an article with one of its keywords, and counting is easy. The `author` column of `references` has the same problem: some references list several authors in one cell, separated by `"; "`. The table `reference_author` is its tidy version, one row per reference and author.

### ✏️ Exercise 4 · Tidy keywords

Using `keywords`, compute `n_pairs`, the number of article–keyword pairs; `n_distinct`, the number of different keywords; and `top`, the keyword used most often.

In [ ]:
n_pairs = ...
n_distinct = ...
top = ...

check.ex4(n_pairs, n_distinct, top)

<details>
<summary>💡 Hint</summary>

The same tools as in Exercise 1: `len`, `.nunique()`. `keywords.value_counts()` lists the keywords from the most to the least used, and `.index[0]` picks the first name.

</details>

<details>
<summary>✅ Solution</summary>

```python
n_pairs = len(keywords)
n_distinct = keywords.nunique()
top = keywords.value_counts().index[0]

check.ex4(n_pairs, n_distinct, top)
```

</details>

## 5 · Population and sample: a first look

A question is always about some set of units: the **population**. Often we can observe only some of them: a **sample**. When we observe every unit of the population, the data are a *census*.

The same table can be a population or a sample, depending on the question. If you ask "what share of *Bibliothecae\.it*'s articles are in English?", the 321 articles are the whole population (assuming that the data list every article), and the answer is simply a count. If you ask "do Italian journals of library science publish more in English than they used to?", the same 321 articles are a sample of a much larger population, and a single journal may be very different from the others.

A sample is useful when it resembles its population. The safest way to get one is to choose the units at random, which is what most of Parts 2 to 4 are about. A sample gathered because it was easy to get, a *convenience sample*, may differ from its population in ways nobody planned. The references in these tables are one: they come only from articles published from 2016 on, and not from all of them.

### ✏️ Exercise 5 · Population or sample?

For each question, are the 321 articles of the journal the population (P) or a sample (S)?

- **A** What share of the articles of *Bibliothecae\.it* are in English?
- **B** Do Italian journals of library science publish more in English than they used to?
- **C** How often do humanities scholars cite digital editions?
- **D** Which author has written most often for *Bibliothecae\.it*?

Set `answer` to four letters, for example `"PPPP"`.

In [ ]:
answer = ...

check.ex5(answer)

<details>
<summary>💡 Hint</summary>

For each question, ask which units it is about. If they are exactly the journal's articles, the table holds all of them. If they are more, the table holds only some.

</details>

<details>
<summary>✅ Solution</summary>

```python
answer = "PSSP"

check.ex5(answer)
```

B and C are about many journals and many scholars, and the journal's articles are a small sample of them, chosen not at random but because they are the ones in these tables.

</details>

## 6 · Measuring: operationalisation, validity and reliability

Many things worth studying cannot be observed directly: a scholar's influence, the quality of an edition, the internationalisation of a journal, the reuse of a dataset. To study them, we choose something observable that stands in for them, and a rule for turning it into values. In Stevens' words, measurement is "the assignment of numerals to objects or events according to rules". Choosing the rule is called **operationalisation**: influence becomes the number of citations; internationalisation becomes the share of articles not in Italian (S0.1).

Every operationalisation can fail in two ways:

- **Validity:** does the measure capture the concept? A citation count measures how often a work is cited, which is related to influence but is not the same thing: works are cited to be criticised, or out of habit, and influential works go uncited.
- **Reliability:** does the measure give the same result when it is repeated, by another person, another day, another run of the program? Two people classifying the same references as "digital objects" or not will disagree on some of them.

A bathroom scale that always shows 2 kg too much is reliable but not valid: it repeats the same wrong answer. A scale whose reading jumps around the right weight is valid on average but unreliable. The four combinations look like this, with each shot a measurement and the centre the true value:

In [ ]:
rng = np.random.default_rng(3)
cases = [("reliable,\nvalid", 0.12, (0, 0)),
         ("reliable,\nnot valid", 0.12, (0.55, 0.45)),
         ("valid on average,\nnot reliable", 0.45, (0, 0)),
         ("neither reliable\nnor valid", 0.45, (0.5, -0.45))]
fig, axes = plt.subplots(1, 4, figsize=(10, 2.8))
for ax, (title, spread, (cx, cy)) in zip(axes, cases):
    for r in (1, 0.66, 0.33):
        ax.add_patch(plt.Circle((0, 0), r, fill=False, color=plots.MUTED))
    shots = rng.normal([cx, cy], spread, size=(10, 2))
    ax.plot(shots[:, 0], shots[:, 1], "o", color=plots.BLUE, ms=4)
    ax.plot(0, 0, "+", color=plots.ORANGE, ms=12, mew=2)
    ax.set(xlim=(-1.3, 1.3), ylim=(-1.3, 1.3), title=title, aspect="equal")
    ax.axis("off")
plt.show()

**In the data.** The `date` column of `references` was not typed by a person. A program read each reference and kept a four-digit number from it as the year of the cited work. Run it again and it gives the same years: it is perfectly reliable. But look at the references whose "year" is after 2025:

In [ ]:
ref_year = pd.to_numeric(bib.references["date"], errors="coerce")
for text, y in bib.references.loc[ref_year > 2025, ["content", "date"]].values[:3]:
    at = text.find(y)                     # where the "year" was found
    print(f"{y}  ←  …{text[max(0, at - 60):at + 15]}…\n")

The program took four digits from a DOI or a web address, or a page number, not a year. For these references the measure is not valid: it measures "some four-digit number in the reference", not "the year the work was published". Every result built on these years, such as the age of the works cited, inherits the error unless someone looks.

### ✏️ Exercise 6 · Years that cannot be years

Using `ref_year` (the years as numbers, with `NaN` where there was no year) and `bib.references`, compute `n_missing`, the number of references with no year at all (an empty `date`); `n_impossible`, the number whose year is before 1400 or after 2025; and `share_unusable`, the share of all references whose year is either missing or impossible.

In [ ]:
n_missing = ...
n_impossible = ...
share_unusable = ...

check.ex6(n_missing, n_impossible, share_unusable)

<details>
<summary>💡 Hint</summary>

`(bib.references["date"] == "").sum()` counts the empty dates. For the impossible years, combine two conditions with `|`, which means "or": `(ref_year < 1400) | (ref_year > 2025)`, and add up the result. The share is the sum of the two counts divided by the number of references.

</details>

<details>
<summary>✅ Solution</summary>

```python
n_missing = (bib.references["date"] == "").sum()
n_impossible = ((ref_year < 1400) | (ref_year > 2025)).sum()
share_unusable = (n_missing + n_impossible) / len(bib.references)

check.ex6(n_missing, n_impossible, share_unusable)
```

About one reference in eight has no usable year. And the rule used here is generous: a year can also be impossible because it comes after the article that cites it, which only the article's year can reveal (`bibliothecae.cited_years` applies that rule too).

</details>

> **🔬 Metascience.** Bibliometric indicators are operationalisations: a citation count stands in for influence, a journal's reputation for the quality of what it publishes. The Leiden Manifesto ([Hicks et al., 2015](https://doi.org/10.1038/520429a)) sets out ten principles for using such indicators in research evaluation, and several of them are about measurement: "Keep data collection and analytical processes open, transparent, and simple", "Avoid misplaced concreteness and false precision", and "Scrutinize indicators regularly and update them". The reference years above show why: an indicator computed from them, such as the median age of the works cited, would look precise, although one reference in eight has no usable year.

## 7 · Missing values, and why they are missing

Data are rarely complete, and missing values hide in several forms:

- **An empty cell.** In the course's tables an empty value is an empty string, `""`. After conversion to numbers it becomes `NaN`. 62 articles have no abstract, and 503 references have no year.
- **A missing row.** Some things are missing without leaving any empty cell. Every article should have at least one author in `article_authors`, yet 68 articles have none: for most of them, `article_author` names their authors, but those authors have no row of their own. Every author who does have a row has both a sex and an affiliation recorded, a hint that the query that built the table may have kept only authors with both, and silently dropped the others. Missing rows show up only when you link tables or count what should be there.
- **A stand-in value.** Sometimes a 0, a "n.d." or a "9999" stands for "unknown". Such values are the most dangerous, because they look like data. In S0.3, articles from 2012–2015 seemed to have 0 references each, because their reference lists were never collected: "not collected" had become "none".

The share of missing values in a variable is an indicator formula, as in S0.1:

$$\text{share missing} = \frac{1}{n} \sum_{i=1}^{n} \mathbb{1}[x_i \text{ is missing}]$$

Read aloud: *"the share missing is one over n times the sum, over all observations, of one if the value is missing and zero if not."* **By hand:** if 2 of 8 articles have no abstract, the share missing is 2/8 = 0.25.

A missingness map draws every empty cell of a table as a dark mark. Here are the 321 articles, from the oldest on the left to the newest on the right:

In [ ]:
empty = (bib.articles.sort_values("date") == "")      # True where a cell is empty
year = bib.articles["date"].str[:4].astype(int)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4),
                         gridspec_kw={"width_ratios": [2, 1]})
axes[0].imshow(empty.T.to_numpy(), aspect="auto", cmap="Greys",
               interpolation="nearest")
axes[0].set_yticks(range(empty.shape[1]))
axes[0].set_yticklabels(empty.columns)
axes[0].set_xlabel("articles, from the oldest to the newest")
no_abstract = (bib.articles["abstract"] == "").groupby(year).mean()
axes[1].bar(no_abstract.index, no_abstract.to_numpy(), color=plots.BLUE)
axes[1].set(xlabel="year", ylabel="share with no abstract")
fig.tight_layout()
plt.show()
print(empty.sum().loc[lambda s: s > 0])

Only the `abstract` column has empty cells, and they are not spread evenly: most belong to 2013–2015. That pattern is the key question about any missing value: **why** is it missing? Statisticians distinguish three answers, named by Donald Rubin ([Rubin, 1976](https://doi.org/10.1093/biomet/63.3.581)):

| Name | The value is missing because of | Example | Is a summary of the rest biased? |
| --- | --- | --- | --- |
| missing completely at random | nothing related to the data: chance | a coffee stain on a page of the register | no, only less precise |
| missing at random | something else that is recorded | abstracts missing more often in early years | yes, unless you take that something into account |
| missing not at random | the missing value itself | long articles more likely to lack a page count | yes, and the data alone cannot show it |

The second name is confusing: "missing at random" means "random once you take the other recorded variables into account", not "random". What matters is the third case: when the chance that a value is missing depends on the value, the values that remain are no longer typical. Move the sliders to hide some page counts, either at random (bias 0) or preferring the longest articles:

In [ ]:
first_last = bib.articles["pages"].str.split("-", expand=True).astype(int)
page_counts = (first_last[1] - first_last[0] + 1).to_numpy()   # as in Exercise 3

def hide_values(share, bias):
    rng = np.random.default_rng(10)                  # the same draw at each move
    weights = page_counts.astype(float) ** bias      # bias 0: all articles alike
    k = int(share / 100 * len(page_counts))
    hidden = rng.choice(len(page_counts), size=k, replace=False,
                        p=weights / weights.sum())
    kept = np.delete(page_counts, hidden)
    fig, ax = plt.subplots(figsize=(6.4, 3.2))
    bins = np.arange(0, 210, 10)
    ax.hist(page_counts, bins=bins, color=plots.MUTED, label="all articles")
    ax.hist(kept, bins=bins, color=plots.BLUE, label="articles not hidden")
    ax.set(xlabel="pages", ylabel="articles")
    ax.legend()
    plt.show()
    print(f"average pages: all articles {page_counts.mean():.1f}, "
          f"articles not hidden {kept.mean():.1f}")

explore(hide_values, share=(0, 60, 5, 40, "share hidden (%)"),
        bias=(0.0, 3.0, 0.5, 0.0, "bias towards long articles"))

Each bar counts the articles in a band of 10 pages. With bias 0, hiding even 40% of the values leaves an average close to the true one, 28.2 pages: it moves a little with the luck of the draw, but in no particular direction. With a strong bias, the average of what remains drops towards 17 pages, and nothing in the remaining values warns you.

### ✏️ Exercise 7 · Missing at random?

If reference years were missing completely at random, they would be missing about equally often for every kind of reference. Compare references that name an author with references that do not (their `author` is empty). Compute `missing_with`, the share of references with no year among those with an author; `missing_without`, the same share among those without an author; and `times`, how many times larger the second share is than the first.

In [ ]:
no_year = bib.references["date"] == ""
has_author = bib.references["author"] != ""
missing_with = ...
missing_without = ...
times = ...

check.ex7(missing_with, missing_without, times)

<details>
<summary>💡 Hint</summary>

`no_year[has_author]` keeps the values of `no_year` for the references with an author, and `.mean()` of True/False values is the share of True. The sign `~` means "not": `no_year[~has_author]` keeps the references without an author.

</details>

<details>
<summary>✅ Solution</summary>

```python
no_year = bib.references["date"] == ""
has_author = bib.references["author"] != ""
missing_with = no_year[has_author].mean()
missing_without = no_year[~has_author].mean()
times = missing_without / missing_with

check.ex7(missing_with, missing_without, times)
```

References without an author lack a year about four times as often. Many of them are early printed books listed by their title, archival documents, web pages, or pieces of references that were split badly. The years are not missing completely at random.

</details>

There is a clue to *why* so many years are missing. The program looked for four digits, but early printed books often give their year in Roman numerals, as in "M. DC. LXX." for 1670. A rough pattern finds such years:

In [ ]:
no_year = bib.references["date"] == ""
roman = bib.references["content"].str.contains(r"\bM\.?\s?D[\.\s]?[CLXVI]")
print(f"{roman.sum()} references seem to give a year in Roman numerals;")
print(f"{no_year[roman].mean():.0%} of them have no year, against "
      f"{no_year.mean():.0%} of all references")

So the oldest works are the most likely to lose their year: whether a year is missing depends on the year itself. Those years are missing not at random, and any summary of the years that remain, such as the age of the works the journal cites, comes out too young.

> **🤖 Why this matters for AI.** Every dataset used to train or test a model has the anatomy of this lesson: a unit (a document, a sentence, a token), variables (the features a model reads) and often a *label*, which is a measurement made by people. When annotators mark whether a sentence is ironic, or whether a reference points to a dataset, they operationalise a concept, and their labels have a validity and a reliability of their own; lesson S3.7 measures how often annotators agree. Module M2 treats a language model itself as an annotator, with exactly the questions of section 6: what does its output measure, and would it give the same answer twice? Missing data matters there too. What was never digitised, written down or collected is absent from training data, and it is never missing completely at random: a model learns the world that its data happen to contain.

## 8 · Wrap-up

### ✍️ Explain it

Your PhD asks how "reuse" of scholarly digital objects can be defined and tracked. In three or four sentences, operationalise it: name one observable indicator of the reuse of a dataset, one way in which the indicator could be invalid, and one way in which it could be unreliable. Then say which values would be missing, and whether they would be missing at random.

Write your answer in the next cell.

*Your answer:*

<details>
<summary>A model answer</summary>

Indicator: the number of articles whose reference lists cite the dataset's DOI. It can be invalid, because a paper may cite a dataset without using it (to acknowledge it, or to criticise it), and many papers use datasets without citing them formally, mentioning them only in the text or not at all. It can be unreliable, because a reference parser may miss DOIs written in unusual forms, so that two parsers, or two versions of one, give different counts. The missing cases are the uncited uses, and they are not missing at random: they are more common in fields and periods with weaker data-citation habits, which may be exactly the humanities.

</details>

### The words and symbols of this lesson

| Term or symbol | Means |
| --- | --- |
| unit of observation, observation | what one row stands for; one row |
| variable, value | a column; one cell |
| $x_{ij}$, $n \times p$ | the value in row i, column j; a table of n rows and p columns |
| nominal, ordinal, interval, ratio | categories; ordered categories; equal steps; a true zero |
| discrete, continuous | separate values such as counts; any value in a range |
| tidy | each variable a column, each observation a row, each type of unit a table |
| population, sample, census | the units a question is about; those observed; all of them observed |
| operationalisation, validity, reliability | the rule that turns a concept into values; measuring the right thing; measuring it consistently |
| `NaN` | "not a number": pandas' mark for a missing number |
| MCAR, MAR, MNAR | missing completely at random, at random, not at random |

### Check your progress

In [ ]:
check.summary();

### Done when

- [ ] Exercises 1 to 7 show ✅ in `check.summary()`.
- [ ] You can say what one row stands for in each of the seven tables.
- [ ] You can name the type of any column of the articles table, and say why.
- [ ] You can explain, with an example from the data, the difference between a reliable and a valid measure.
- [ ] You have written down one operationalisation of "reuse" for your PhD, with its weak points.

### Further reading

- Wickham, H. (2014). Tidy data. *Journal of Statistical Software*, 59(10). Open access: <https://doi.org/10.18637/jss.v059.i10>
- Stevens, S. S. (1946). On the theory of scales of measurement. *Science*, 103(2684), 677–680. <https://doi.org/10.1126/science.103.2684.677>
- Hicks, D., Wouters, P., Waltman, L., de Rijcke, S., & Rafols, I. (2015). Bibliometrics: The Leiden Manifesto for research metrics. *Nature*, 520(7548), 429–431. <https://doi.org/10.1038/520429a>

**Next:** S1.2 · Counting and tables.